# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hardik144/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### My lane as an ML task

My chosen lane is Google Search Ranking & Discoverability. I will treat this as a ranking and scoring problem. The goal is to use available content and search-performance data to understand which pages may need attention and to estimate their expected search click-through rate (CTR). The output can help a content team prioritize pages for review and improvement. I chose this lane because the starter dataset contains search-performance information that can support an initial analysis. This is a provisional choice and may change as I explore the data.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target or proxy

My initial target is the observed click-through rate (CTR) of a content page. CTR is a useful proxy for how often a page receives clicks relative to the impressions it receives. I will investigate whether available variables, such as average search position, search volume and competition, help explain differences in CTR.

The target is a proxy for search performance, not a direct measure of content quality. A page with a low CTR may have several possible explanations, and the available data may not contain all the information needed to identify them.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success metric

For an initial CTR prediction model, I will use Mean Absolute Error (MAE) to measure the average absolute difference between predicted CTR and observed CTR. A lower MAE indicates smaller prediction errors.

I will compare the model against a simple baseline that predicts the training-set mean CTR. If the task is evaluated as a ranking problem, I will also consider a ranking metric such as Normalized Discounted Cumulative Gain (NDCG), where higher values indicate that relevant pages are placed nearer the top of the ranking.

I will use a held-out test set to evaluate performance and will not assume that a model is useful until it is compared with the baseline.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### The unit of analysis

The unit of analysis is one content page. Each row in my analysis dataframe represents one page, identified by its content ID, along with the available search-performance variables. The observed CTR will serve as the target proxy. I will inspect the actual dataset before deciding which features can be used and will check for missing values and duplicate page records.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [8]:

from pathlib import Path
import pandas as pd
import numpy as np

# 1. Find and load the starter dataset
matches = list(Path(".").rglob("content_refresh_anonymized.csv"))

if not matches:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv. "
        "Check that the starter dataset is available in Colab."
    )

csv_path = matches[0]
df = pd.read_csv(csv_path)

print("Dataset path:", csv_path)
print("Dataset shape:", df.shape)

# 2. Inspect the column names
print("\nAvailable columns:")
print(df.columns.tolist())

# 3. Identify the columns needed for this lane
required = ["content_id", "avg_position", "ctr"]
missing = [col for col in required if col not in df.columns]

if missing:
    raise ValueError(
        f"Missing expected columns: {missing}. "
        "Check the printed column names and update the code."
    )

# 4. Select available explanatory variables
candidate_features = [
    "avg_position",
    "search_volume",
    "competition"
]

feature_columns = [
    col for col in candidate_features if col in df.columns
]

# 5. Create the page-level analysis dataframe
analysis_df = df[
    ["content_id"] + feature_columns + ["ctr"]
].copy()

analysis_df = analysis_df.rename(
    columns={"ctr": "target_ctr"}
)

# 6. Show the unit of analysis
print("\nAnalysis dataframe shape:", analysis_df.shape)
print("One row represents one content page.")
display(analysis_df.head())

# 7. Check the data
print("\nMissing values:")
print(analysis_df.isnull().sum())

print("\nDuplicate content IDs:",
      analysis_df["content_id"].duplicated().sum())

# 8. Sketch the target column
print("\nTarget (CTR proxy) summary:")
print(analysis_df["target_ctr"].describe())

# Optional exploratory label: pages at or above the median CTR
median_ctr = analysis_df["target_ctr"].median()

analysis_df["high_ctr_proxy"] = (
    analysis_df["target_ctr"] >= median_ctr
).astype(int)

print("\nMedian CTR:", median_ctr)
print("Target label: 1 = CTR at or above median, 0 = below median")

display(
    analysis_df[
        ["content_id", "target_ctr", "high_ctr_proxy"]
    ].head()
)

Dataset path: content_refresh_anonymized.csv
Dataset shape: (30000, 44)

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Analysis dataframe shape: (30000, 5)
One row represents one content page.


,content_id,avg_position,search_volume,competition,target_ctr
0,content_304f48230142,10.6,10.0,0.67,0.76
1,content_a1fb4e703a9e,20.3,90.0,0.01,0.05
2,content_9aa793d4d895,36.5,0.0,0.00,0.09
3,content_331d6c4de07b,6.2,10.0,0.00,0.49
4,content_d99b7a2d90ca,44.0,0.0,0.00,0.13



Missing values:
content_id          0
avg_position        0
search_volume    2468
competition      2468
target_ctr          0
dtype: int64

Duplicate content IDs: 0

Target (CTR proxy) summary:
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: target_ctr, dtype: float64

Median CTR: 0.07
Target label: 1 = CTR at or above median, 0 = below median


,content_id,target_ctr,high_ctr_proxy
0,content_304f48230142,0.76,1
1,content_a1fb4e703a9e,0.05,0
2,content_9aa793d4d895,0.09,1
3,content_331d6c4de07b,0.49,1
4,content_d99b7a2d90ca,0.13,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML may help more than a fixed rule

A fixed rule could flag every page whose CTR is below a chosen threshold. This is simple, but it does not account for differences between pages, such as search position, search volume and competition. A machine learning model may be able to combine several variables and estimate CTR more flexibly than a single threshold.

However, ML is not automatically better than a fixed rule. I will compare any model with a simple baseline using held-out data. If the model does not improve on the baseline or its results are unreliable, a simpler rule may be more appropriate. The model output would support human review and prioritization, not automatically determine which pages must be changed.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.